# **Tarea Calificada 2**

**Curso:** Fundamentos de Programación para IA Generativa Aplicada a la Ciencias Sociales y la Gestión Pública

**Profesor:** Alfonso Rodriguez Saldarriaga

**Email:** alfonso.rodriguezs@pucp.pe

**Integrantes:** Katia Villalobos Carlos, Ariana Rodriguez Ramirez y Diego Canales Becerra


---

# 3. Carga y limpieza de datos

## 3.1 Fuentes de datos

In [ ]:
## Setup — librerías
import os, zipfile, requests
import geopandas as gpd
import pandas as pd
import io
from google.colab import auth
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf
import matplotlib.cm as cm
from matplotlib.lines import Line2D

**Encuesta Nacional de Hogares (2020-2024)**

La Encuesta Nacional de Hogares (ENAHO) es uno de los instrumentos estadístico más importantes del Perú, ejecutado de manera continua por el Instituto Nacional de Estadística e Informática (INEI). Su objetivo principal es permitir el seguimiento de las condiciones de vida, la medición de la pobreza monetaria y multidimensional, y el análisis del impacto de las políticas sociales en la población peruana.

## 3.2 Módulo 400 — Salud (`salud`)

| Variable | Descripción | Categorías |
|----------|------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| P401     | Padece enfermedad o malestar crónico | Sí = 1, No = 0 |
| P4021    | Síntoma o malestar (tos, dolor)    | Sí = 1, No = 0 (En las últimas 4 semanas) |
| P4022    | Enfermedad (gripe, colitis, etc.)  | Sí = 1, No = 0 (En las últimas 4 semanas) |
| P4023    | Recaída de enfermedad crónica      | Sí = 1, No = 0 (En las últimas 4 semanas) |
| P4024    | Accidente                          | Sí = 1, No = 0 (En las últimas 4 semanas) |
| P4025    | No tuvo enfermedad ni síntoma      | Sí = 1, No = 0 (En las últimas 4 semanas) |
| P4026    | Síntoma del COVID-19               | Sí = 1, No = 0 (En las últimas 4 semanas) |
| P4191    | Afiliado a: ESSALUD                | Sí = 1, No = 0 |
| P4192    | Afiliado a: Seguro Integral de Salud (SIS) | Sí = 1, No = 0 |
| P4193    | Afiliado a: Seguro de Fuerzas Armadas/Policiales | Sí = 1, No = 0 |
| P4194    | Afiliado a: Seguro Privado de Salud | Sí = 1, No = 0 |
| P4195    | Afiliado a: Entidad Prestadora de Salud (EPS) | Sí = 1, No = 0 |
| P4196    | Afiliado a: Seguro Universitario   | Sí = 1, No = 0 |
| P4197    | Afiliado a: Seguro Escolar Privado | Sí = 1, No = 0 |
| P4198    | Afiliado a: Otro seguro            | Sí = 1, No = 0 |
| P4151_01 | Gasto pagado por miembro: Consulta | Sí = 1, No = 0 |
| P4151_02 | Gasto pagado: Medicinas/Insumos    | Sí = 1, No = 0 |
| P4151_03 | Gasto pagado: Análisis             | Sí = 1, No = 0 |
| P4151_04 | Gasto pagado: Rayos X/Tomografía   | Sí = 1, No = 0 |
| P4151_05 | Gasto pagado: Otros exámenes       | Sí = 1, No = 0 |
| P4151_06 | Gasto pagado: Servicio dental      | Sí = 1, No = 0 |
| P4151_07 | Gasto pagado: Servicio oftalmológico| Sí = 1, No = 0 |
| P4151_08 | Gasto pagado: Compra de lentes     | Sí = 1, No = 0 |
| P4151_09 | Gasto pagado: Vacunas              | Sí = 1, No = 0 |
| I41601   | Monto anual: Consulta              | Numérica (Soles anuales imputados) |
| I41602   | Monto anual: Medicinas/Insumos     | Numérica (Soles anuales imputados) |
| I41603   | Monto anual: Análisis              | Numérica (Soles anuales imputados) |
| I41604   | Monto anual: Rayos X/Tomografía    | Numérica (Soles anuales imputados) |
| I41605   | Monto anual: Otros exámenes        | Numérica (Soles anuales imputados) |
| I41606   | Monto anual: Servicio dental       | Numérica (Soles anuales imputados) |
| I41607   | Monto anual: Servicio oftalmológico| Numérica (Soles anuales imputados) |
| I41608   | Monto anual: Compra de lentes      | Numérica (Soles anuales imputados) |
| I41609   | Monto anual: Vacunas               | Numérica (Soles anuales imputados) |

In [ ]:
# 1. Autenticación
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive_api = GoogleDrive(gauth)

# 2. Diccionario con IDs
archivos_salud = {
    "2024": "1pCYDLSWoGnmRHQBB2bkMju9IpBEdsqdj",
    "2023": "1TgFPNr8D_wBCvPint0daDPK-JaJDkxLZ",
    "2022": "1a7gY0Wdr8nX0yPaJf_hY_qYF53rzhXdI",
    "2021": "1QVx_ZPx3MOJ45hk4FutEPRi0pR8tIMAs",
    "2020": "1CaZx2U-1zpa_isTe-5ZA1i0MjReKSsF8"
}

lista_años = []

# 3. Bucle de descarga y procesamiento
for anio, file_id in archivos_salud.items():
    print(f"--- Descargando año {anio} ---")

    try:
        archivo_descargado = drive_api.CreateFile({'id': file_id})
        # Usamos latin-1 porque ENAHO suele tener caracteres especiales
        contenido = archivo_descargado.GetContentString(encoding='latin-1')

        # Leer el contenido como CSV
        df = pd.read_csv(io.StringIO(contenido), low_memory=False)

        # Normalizar nombres de columnas a minúsculas
        df.columns = df.columns.str.lower()

        # 4. Selección de variables solicitadas
        cols_fijas = ['conglome', 'vivienda', 'hogar', 'ubigeo', 'codperso', 'p4021',
                      'p4022', 'p4023', 'p4024', 'p4025', 'p4026', 'p401',
                      'p4191', 'p4192', 'p4193', 'p4194', 'p4195',
                      'p4196', 'p4197', 'p4198','factor07']

        # Filtrar variables con patrón i41601* y p4151_*
        cols_patron = [c for c in df.columns if c.startswith('i4160') or c.startswith('p4151')]

        # Unir y asegurar que solo busquemos las que existen en el DF
        todas_las_cols = [c for c in (cols_fijas + cols_patron) if c in df.columns]

        # Crear copia filtrada y añadir la variable año
        df_final = df[todas_las_cols].copy()
        df_final['año'] = anio

        lista_años.append(df_final)
        print(f"Éxito: {anio} procesado con {len(df_final)} filas.")

    except Exception as e:
        print(f"Error al procesar el año {anio}: {e}")

# 5. Consolidar todo
if lista_años:
    enaho_salud_total = pd.concat(lista_años, ignore_index=True)
    print("\n" + "="*30)
    print("¡PROCESO COMPLETADO!")
    print(f"Dimensiones finales: {enaho_salud_total.shape}")
    display(enaho_salud_total.head())
else:
    print("No se pudo descargar ninguna base de datos.")

--- Descargando año 2024 ---
Éxito: 2024 procesado con 110451 filas.
--- Descargando año 2023 ---
Éxito: 2023 procesado con 112530 filas.
--- Descargando año 2022 ---
Éxito: 2022 procesado con 114709 filas.
--- Descargando año 2021 ---
Éxito: 2021 procesado con 114239 filas.
--- Descargando año 2020 ---
Éxito: 2020 procesado con 120346 filas.

¡PROCESO COMPLETADO!
Dimensiones finales: (572275, 111)


,conglome,vivienda,hogar,ubigeo,codperso,p4021,p4022,p4023,p4024,p4025,...,p4151_h13,p41510h01,p41510h06,p41510h13,p41510_h01,p41510_h06,p41510_h13,p41511_h01,p41511_h06,p41511_h13
0,15009,13,11,10101,1,0,0,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,15009,13,11,10101,2,1,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,15009,47,11,10101,1,1,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,15009,47,11,10101,2,1,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,15009,47,11,10101,3,0,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
enaho_salud_total.columns

Index(['conglome', 'vivienda', 'hogar', 'ubigeo', 'codperso', 'p4021', 'p4022',
       'p4023', 'p4024', 'p4025',
       ...
       'p4151_h13', 'p41510h01', 'p41510h06', 'p41510h13', 'p41510_h01',
       'p41510_h06', 'p41510_h13', 'p41511_h01', 'p41511_h06', 'p41511_h13'],
      dtype='object', length=111)

In [ ]:
# Copia de trabajo
df_salud = enaho_salud_total.copy()

# 1. Identificar todas las columnas de interés
cols_monto_all = [c for c in df_salud.columns if c.startswith('i416')]
cols_pago_all = [c for c in df_salud.columns if c.startswith('p4151')]
cols_seguro_all = [c for c in df_salud.columns if c.startswith('p419')]
cols_morbilidad = ['p4021', 'p4022', 'p4023', 'p4024', 'p4025', 'p4026', 'p401']

# 2. Conversión masiva a numérico
# Esto elimina espacios en blanco o textos que impiden la suma
for col in (cols_monto_all + cols_pago_all + cols_seguro_all + cols_morbilidad):
    if col in df_salud.columns:
        df_salud[col] = pd.to_numeric(df_salud[col], errors='coerce').fillna(0)

print(f"Limpieza profunda terminada. Columnas de gasto detectadas: {len(cols_monto_all)}")

Limpieza profunda terminada. Columnas de gasto detectadas: 9


In [ ]:
# 2.1. Variable Seguro
df_salud['tiene_seguro'] = (df_salud[cols_seguro_all] == 1).any(axis=1).astype(int)

# 2.2. Variable Enfermo (4 semanas)
cols_enf = ['p4021', 'p4022', 'p4023', 'p4024', 'p4026']
df_salud['enfermo_4sem'] = (df_salud[cols_enf] == 1).any(axis=1).astype(int)

# 2.3. Variable Gasto de Bolsillo (Lógica Robusta)
# Vamos a crear una lista de los montos que SI fueron pagados por el hogar
df_salud['gasto_bolsillo'] = 0

for i in range(1, 17):
    suffix = str(i).zfill(2) # "01", "02"...

    # Buscamos el nombre de la columna de monto
    col_m = [c for c in cols_monto_all if suffix in c or (i > 9 and str(i) in c)]
    col_p = [c for c in cols_pago_all if suffix in c or (i > 9 and str(i) in c)]

    if col_m and col_p:
        # Usamos el primer match encontrado para cada rubro
        m = col_m[0]
        p = col_p[0]
        # Sumar al total si el pago es 1 (miembro del hogar)
        df_salud['gasto_bolsillo'] += np.where(df_salud[p] == 1, df_salud[m], 0)

# --- REPORTE DE CONTROL ---
enaho_salud_total = df_salud

print("--- Resumen Final ---")
print(f"Total registros: {len(enaho_salud_total)}")
print(f"% con Seguro: {enaho_salud_total['tiene_seguro'].mean():.2%}")
print(f"% Enfermos: {enaho_salud_total['enfermo_4sem'].mean():.2%}")

# Verificamos si hay al menos una fila con gasto > 0
gastos_positivos = enaho_salud_total[enaho_salud_total['gasto_bolsillo'] > 0]
if not gastos_positivos.empty:
    print(f"Gasto bolsillo promedio (S/.): {gastos_positivos['gasto_bolsillo'].mean():.2f}")
    print(f"Máximo gasto encontrado: S/. {enaho_salud_total['gasto_bolsillo'].max():.2f}")
else:
    print("ALERTA: El gasto sigue siendo 0. Revisando estructura...")
    # Esto nos dirá si las columnas p4151 tienen valores distintos a 1
    if len(cols_pago_all) > 0:
        print(f"Valores encontrados en {cols_pago_all[0]}: {enaho_salud_total[cols_pago_all[0]].unique()}")

--- Resumen Final ---
Total registros: 572275
% con Seguro: 86.15%
% Enfermos: 52.35%
Gasto bolsillo promedio (S/.): 865.81
Máximo gasto encontrado: S/. 73509.00


/tmp/ipykernel_5655/2386874278.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_salud['tiene_seguro'] = (df_salud[cols_seguro_all] == 1).any(axis=1).astype(int)
/tmp/ipykernel_5655/2386874278.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_salud['enfermo_4sem'] = (df_salud[cols_enf] == 1).any(axis=1).astype(int)
/tmp/ipykernel_5655/2386874278.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all c

In [ ]:
# 1. Aseguramos que el factor de expansión sea numérico
enaho_salud_total['factor07'] = pd.to_numeric(enaho_salud_total['factor07'], errors='coerce').fillna(0)

# 2. Agrupamos por año para calcular las métricas
resumen_anual = enaho_salud_total.groupby('año').apply(lambda x: pd.Series({
    'Población Total (Est.)': x['factor07'].sum(),
    '% con Seguro': (x['tiene_seguro'] * x['factor07']).sum() / x['factor07'].sum() * 100,
    '% Enfermos (4 sem)': (x['enfermo_4sem'] * x['factor07']).sum() / x['factor07'].sum() * 100,
    'Gasto Bolsillo Total (Millones S/.)': (x['gasto_bolsillo'] * x['factor07']).sum() / 1_000_000,
    'Gasto Promedio por Persona (S/.)': (x['gasto_bolsillo'] * x['factor07']).sum() / x['factor07'].sum()
})).reset_index()

# 3.
print("### TABLA DE VALORES ANUALES (POBLACIÓN EXPANDIDA) ###")
display(resumen_anual)

/tmp/ipykernel_5655/1738693140.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  resumen_anual = enaho_salud_total.groupby('año').apply(lambda x: pd.Series({


### TABLA DE VALORES ANUALES (POBLACIÓN EXPANDIDA) ###


,año,Población Total (Est.),% con Seguro,% Enfermos (4 sem),Gasto Bolsillo Total (Millones S/.),Gasto Promedio por Persona (S/.)
0,2020,3.322160e+07,77.117198,46.187439,5094.889042,153.360719
1,2021,3.350053e+07,81.214463,47.107128,9501.411691,283.619726
2,2022,3.392257e+07,85.868230,52.227826,10828.734422,319.219123
3,2023,3.428533e+07,88.276924,54.843813,11767.755844,343.230052
4,2024,3.464694e+07,90.675871,54.863401,11261.827989,325.045367


---
## 3.3 Módulo 300 — Educación (`edu`)

| Variable | Descripción                        | Categorías                                                                                                                                                                                       ocupación |
|----------|------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| P301A    | Último nivel educativo aprobado    | Sin nivel = 1, Educación inicial = 2, Primaria incompleta = 3, Primaria completa = 4, Secundaria incompleta = 5, Secundaria completa = 6, Sup. no universitaria incompleta = 7, Sup. no universitaria completa = 8, Sup. universitaria incompleta = 9, Sup. universitaria completa = 10, Maestría/Doctorado = 11, Básica especial = 12, Missing = 99 |
| P301B    | Último año dentro del nivel        | Numérica, Missing = 99                                                                                                                                                                           |

In [ ]:
# 1. Autenticación:
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive_api = GoogleDrive(gauth)

# 2. Diccionario con los IDs de Educación
archivos_edu = {
    "2024": "13ccnOThjM0Ee6n45a6xlBcgpwjd4BUVb",
    "2023": "1qKDhIx9-l-3jUg-nohxp3q2nY_cPiQ3b",
    "2022": "18j3bwezT1UGy9le6svbjjhS978ZG26Ty",
    "2021": "1FeH04PnKqW--vaQo-DEe-liCl0ezxZro",
    "2020": "1l8YUebbcCZ0mjB5ZnBANiipC2F2uI_mh"
}

lista_años_edu = []

# 3. Bucle de descarga y procesamiento
for anio, file_id in archivos_edu.items():
    print(f"--- Procesando Educación año {anio} ---")

    try:
        # Descarga
        archivo_descargado = drive_api.CreateFile({'id': file_id})
        contenido = archivo_descargado.GetContentString(encoding='latin-1')

        # Leer como CSV
        df = pd.read_csv(io.StringIO(contenido), low_memory=False)

        # Normalizar nombres de columnas a minúsculas para trabajar internamente
        df.columns = df.columns.str.lower()

        # 4. Selección de variables solicitadas (en minúsculas para el filtro)
        cols_edu = ['ubigeo', 'conglome', 'vivienda', 'hogar', 'codperso', 'p301a', 'p301b']

        # Filtrar solo las existentes
        todas_las_cols = [c for c in cols_edu if c in df.columns]
        edu_sel = df[todas_las_cols].copy()

        # --- LIMPIEZA ESPECÍFICA ---
        # Volvemos a mayúsculas para cumplir con tu estándar solicitado
        edu_sel.columns = edu_sel.columns.str.upper()

        edu_clean = edu_sel.copy()

        # Formatear UBIGEO y crear cod_dpto
        if 'UBIGEO' in edu_clean.columns:
            edu_clean['UBIGEO'] = edu_clean['UBIGEO'].astype(str).str.zfill(6)
            edu_clean['cod_dpto'] = edu_clean['UBIGEO'].str[:2]

        # Limpiar llaves de persona (quitar espacios)
        for col in ['CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO']:
            if col in edu_clean.columns:
                edu_clean[col] = edu_clean[col].astype(str).str.strip()

        # Añadir año
        edu_clean['AÑO'] = anio

        lista_años_edu.append(edu_clean)
        print(f"Éxito: {anio} procesado. Dimensiones: {edu_clean.shape}")

    except Exception as e:
        print(f"Error al procesar el año {anio}: {e}")

# 5. Consolidar todo el módulo de Educación
if lista_años_edu:
    enaho_edu_total = pd.concat(lista_años_edu, ignore_index=True)
    print("\n" + "="*30)
    print("¡PROCESO DE EDUCACIÓN COMPLETADO!")
    print(f"Dimensiones finales totales: {enaho_edu_total.shape}")
    display(enaho_edu_total.head())
else:
    print("No se pudo procesar ninguna base de datos de educación.")

--- Procesando Educación año 2024 ---
Éxito: 2024 procesado. Dimensiones: (106619, 9)
--- Procesando Educación año 2023 ---
Éxito: 2023 procesado. Dimensiones: (108354, 9)
--- Procesando Educación año 2022 ---
Éxito: 2022 procesado. Dimensiones: (110257, 9)
--- Procesando Educación año 2021 ---
Éxito: 2021 procesado. Dimensiones: (109867, 9)
--- Procesando Educación año 2020 ---
Éxito: 2020 procesado. Dimensiones: (115777, 9)

¡PROCESO DE EDUCACIÓN COMPLETADO!
Dimensiones finales totales: (550874, 9)


,UBIGEO,CONGLOME,VIVIENDA,HOGAR,CODPERSO,P301A,P301B,cod_dpto,AÑO
0,010101,15009,13,11,1,11,2,01,2024
1,010101,15009,13,11,2,9,1,01,2024
2,010101,15009,47,11,1,8,3,01,2024
3,010101,15009,47,11,2,9,2,01,2024
4,010101,15009,47,11,3,3,0,01,2024


In [ ]:
# ── 4.2 Variables de educación ────────────────────────────
enaho_edu_total = enaho_edu_total.copy()

# 1. Limpieza y conversión de variables educativas (P301A: Nivel, P301B: Grado)
for col in ['P301A', 'P301B']:
    enaho_edu_total[col] = pd.to_numeric(enaho_edu_total[col], errors='coerce')
    enaho_edu_total[col] = enaho_edu_total[col].replace(99, np.nan)

# 2. calcular años de educación acumulados
def calcular_anios_educ(nivel, anio_nivel):
    """Convierte nivel educativo + año dentro del nivel → años de educación acumulados."""
    if pd.isna(nivel):
        return np.nan
    nivel = int(nivel)
    # Sin nivel / Inicial / Especial = 0
    if nivel in [1, 2, 12]:   return 0
    # Primaria incompleta
    elif nivel == 3:          return anio_nivel if not pd.isna(anio_nivel) else np.nan
    # Primaria completa
    elif nivel == 4:          return 6
    # Secundaria incompleta (6 primaria + años de secundaria)
    elif nivel == 5:          return 6  + anio_nivel if not pd.isna(anio_nivel) else np.nan
    # Secundaria completa
    elif nivel == 6:          return 11
    # Superior No Univ. Incompleta (11 colegio + años superior)
    elif nivel == 7:          return 11 + anio_nivel if not pd.isna(anio_nivel) else np.nan
    # Superior No Univ. Completa
    elif nivel == 8:          return 14
    # Superior Univ. Incompleta (11 colegio + años univ)
    elif nivel == 9:          return 11 + anio_nivel if not pd.isna(anio_nivel) else np.nan
    # Superior Univ. Completa
    elif nivel == 10:         return 16
    # Postgrado
    elif nivel == 11:         return 18
    else:                     return np.nan

# Aplicar la función para crear la variable continua
enaho_edu_total['anios_educ'] = enaho_edu_total.apply(
    lambda r: calcular_anios_educ(r['P301A'], r['P301B']), axis=1
)

# 3. Creación de Dummy educación superior (Nivel 9 o más)
enaho_edu_total['edu_superior'] = (enaho_edu_total['P301A'] >= 9).astype(float)

# 4. Nivel simplificado en 3 categorías
def nivel_3cat(cod):
    if pd.isna(cod):   return np.nan
    elif cod <= 4:     return 'Básica o menos'
    elif cod <= 6:     return 'Secundaria'
    else:              return 'Superior'

enaho_edu_total['nivel_3cat'] = enaho_edu_total['P301A'].apply(nivel_3cat)

# 5. Verificación de resultados
print("Resumen de Años de Educación:")
print(enaho_edu_total['anios_educ'].describe().round(2))
print(f"\nMissings anios_educ: {enaho_edu_total['anios_educ'].isnull().sum():,}")

print("\nDistribución nivel_3cat (limpio):")
print(enaho_edu_total['nivel_3cat'].value_counts(dropna=False))

Resumen de Años de Educación:
count    550443.00
mean          7.48
std           5.52
min           0.00
25%           0.00
50%           8.00
75%          11.00
max          18.00
Name: anios_educ, dtype: float64

Missings anios_educ: 431

Distribución nivel_3cat (limpio):
nivel_3cat
Básica o menos    231843
Secundaria        197377
Superior          121223
NaN                  431
Name: count, dtype: int64


## 3.4 Módulo 34 — Sumaria (`sumaria`)

| Variable | Descripción | Categorías |
|----------|------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| CONGLOME | Número de conglomerado             | Identificador (String) |
| VIVIENDA | Número de selección de vivienda    | Identificador (String) |
| HOGAR    | Número secuencial del hogar        | Identificador (String) |
| UBIGEO   | Ubicación geográfica               | Ubicación (6 dígitos) |
| MIEPERHO | Total de miembros del hogar        | Numérica (Cantidad de personas) |
| INGMO1HD | Ingreso monetario (Bruto)          | Numérica (Soles anuales) |
| INGMO2HD | Ingreso monetario (Neto)           | Numérica (Soles anuales) |
| INGHOG1D | Ingreso bruto total (Monetario + No monetario) | Numérica (Soles anuales) |
| INGHOG2D | Ingreso neto total (Monetario + No monetario) | Numérica (Soles anuales) |
| POBREZA  | Condición de pobreza               | Pobre extremo = 1, Pobre no extremo = 2, No pobre = 3 |
| FACTOR07 | Factor de expansión                | Peso poblacional (Anual proyecciones CPV-2007) |

In [ ]:
# 1. Diccionario con los IDs de Sumaria
archivos_sumaria = {
    "2024": "1oZIlZy8-wStpzmGGM0VnuFfhpu0fFwPI",
    "2023": "1bSqMqE7gFTU8kVoL9ZAHa14R2wQjJsXi",
    "2022": "1Xz-SRraAQoKTQIWY6OQc1DcG0yyNd0cw",
    "2021": "1c3cWbByl6C_smwZ_L4UToDdxX2SdgKws",
    "2020": "1buAFP-ZqFCXiKgH4cyXMhKILbVe8Qaql"
}

lista_años_sumaria = []

# 2. Bucle de descarga y procesamiento
for anio, file_id in archivos_sumaria.items():
    print(f"--- Procesando Sumaria año {anio} ---")

    try:
        # Descarga vía API (pydrive)
        archivo_descargado = drive_api.CreateFile({'id': file_id})
        contenido = archivo_descargado.GetContentString(encoding='latin-1')

        # Leer como CSV
        df = pd.read_csv(io.StringIO(contenido), low_memory=False)

        # Normalizar nombres de columnas a minúsculas para filtrar
        df.columns = df.columns.str.lower()

        # 3. Aplicar el KEEP solicitado
        cols_sumaria = [
            'conglome', 'vivienda', 'hogar', 'ubigeo', 'mieperho', 'pobreza', 'factor07', 'inghog1d', 'inghog2d',
            'ingmo2hd', 'ingmo1hd' ,'codperso'
        ]

        # Filtrar solo las existentes y crear copia
        todas_las_cols = [c for c in cols_sumaria if c in df.columns]
        sumaria_sel = df[todas_las_cols].copy()

        # --- LIMPIEZA Y ESTANDARIZACIÓN ---
        # Pasamos a MAYÚSCULAS para consistencia
        sumaria_sel.columns = sumaria_sel.columns.str.upper()

        # Formatear UBIGEO y llaves
        sumaria_sel['UBIGEO'] = sumaria_sel['UBIGEO'].astype(str).str.zfill(6)
        for col in ['CONGLOME', 'VIVIENDA', 'HOGAR']:
            sumaria_sel[col] = sumaria_sel[col].astype(str).str.strip()

        # Añadir año
        sumaria_sel['AÑO'] = anio

        lista_años_sumaria.append(sumaria_sel)
        print(f"Éxito: {anio} procesado. Registros: {len(sumaria_sel)}")

    except Exception as e:
        print(f"Error al procesar el año {anio}: {e}")

# 4. Consolidar Sumaria
if lista_años_sumaria:
    enaho_sumaria_total = pd.concat(lista_años_sumaria, ignore_index=True)
    print("\n" + "="*30)
    print("¡PROCESO DE SUMARIA COMPLETADO!")
    display(enaho_sumaria_total.head())

--- Procesando Sumaria año 2024 ---
Éxito: 2024 procesado. Registros: 33691
--- Procesando Sumaria año 2023 ---
Éxito: 2023 procesado. Registros: 33886
--- Procesando Sumaria año 2022 ---
Éxito: 2022 procesado. Registros: 34213
--- Procesando Sumaria año 2021 ---
Éxito: 2021 procesado. Registros: 34245
--- Procesando Sumaria año 2020 ---
Éxito: 2020 procesado. Registros: 34490

¡PROCESO DE SUMARIA COMPLETADO!


,CONGLOME,VIVIENDA,HOGAR,UBIGEO,MIEPERHO,POBREZA,FACTOR07,INGHOG1D,INGHOG2D,INGMO2HD,INGMO1HD,AÑO
0,15009,13,11,010101,2,3,79.816757,52162.609375,52162.609375,42204.0,42204.0,2024
1,15009,47,11,010101,3,3,79.816757,42098.042969,40832.042969,26031.0,27297.0,2024
2,15009,59,11,010101,1,3,79.816757,15098.497070,15098.497070,15062.0,15062.0,2024
3,15009,71,11,010101,2,3,79.816757,41865.953125,41082.953125,34251.0,35034.0,2024
4,15009,84,11,010101,5,3,79.816757,47659.160156,47659.160156,34179.0,34179.0,2024


In [ ]:
# Copia de seguridad
df_sumaria = enaho_sumaria_total.copy()

# 1.1. Conversión Masiva a Numérico
# Definimos las columnas de dinero y miembros
cols_dinero = ['INGHOG1D', 'INGHOG2D', 'INGMO1HD', 'INGMO2HD', 'FACTOR07']

for col in cols_dinero:
    if col in df_sumaria.columns:
        # Convertimos a número y lo que no sea número se vuelve NaN
        df_sumaria[col] = pd.to_numeric(df_sumaria[col], errors='coerce')
        # Llenamos nulos con 0 para evitar errores en sumas posteriores
        df_sumaria[col] = df_sumaria[col].fillna(0)

# 1.2. Limpieza de Pobreza
# ENAHO usa 1: Pobre Extremo, 2: Pobre No Extremo, 3: No Pobre
if 'POBREZA' in df_sumaria.columns:
    df_sumaria['POBREZA'] = pd.to_numeric(df_sumaria['POBREZA'], errors='coerce')
    # Creamos una etiqueta más clara (Opcional)
    df_sumaria['ES_POBRE'] = np.where(df_sumaria['POBREZA'] < 3, 1, 0)

print("--- Limpieza de Sumaria Completada ---")

--- Limpieza de Sumaria Completada ---


In [ ]:
# 2.1. Ingreso Mensual del Hogar (La ENAHO viene en valores ANUALES)
# Usamos INGHOG2D (Ingreso Neto Total) que es el estándar para bienestar
df_sumaria['ingreso_mensual'] = df_sumaria['INGHOG2D'] / 12

# 2.2. Ingreso Per Cápita Mensual (Ingreso entre número de miembros)
df_sumaria['ingreso_pcapita'] = (df_sumaria['INGHOG2D'] / df_sumaria['MIEPERHO']) / 12

# 2.3. Clasificación por Departamento (vía UBIGEO)
# Ya lo tenías en Educación, pero lo aseguramos aquí para análisis regional
df_sumaria['DEPARTAMENTO'] = df_sumaria['UBIGEO'].str[:2]

# --- REPORTE DE CONTROL ---
enaho_sumaria_total = df_sumaria

print("--- Resumen de Sumaria Limpia ---")
print(f"Ingreso mensual promedio nacional: S/. {enaho_sumaria_total['ingreso_mensual'].mean():,.2f}")
print(f"Tasa de pobreza (Muestral): {enaho_sumaria_total['ES_POBRE'].mean():.2%}")

--- Resumen de Sumaria Limpia ---
Ingreso mensual promedio nacional: S/. 2,855.03
Tasa de pobreza (Muestral): 20.64%


In [ ]:
# 1. Aseguramos que las variables críticas sean numéricas para el cálculo
cols_anuales = ['INGHOG2D', 'INGMO2HD', 'FACTOR07', 'MIEPERHO', 'ES_POBRE']
for col in cols_anuales:
    if col in enaho_sumaria_total.columns:
        enaho_sumaria_total[col] = pd.to_numeric(enaho_sumaria_total[col], errors='coerce').fillna(0)

# 2. Creación de la Tabla Anual Ponderada
resumen_sumaria_anual = enaho_sumaria_total.groupby('AÑO').apply(lambda x: pd.Series({
    'Hogares Totales (Est.)': x['FACTOR07'].sum(),
    'Ingreso Neto Anual S/. (Promedio)': (x['INGHOG2D'] * x['FACTOR07']).sum() / x['FACTOR07'].sum(),
    'Ingreso Neto Mensual S/. (Promedio)': ((x['INGHOG2D'] * x['FACTOR07']).sum() / x['FACTOR07'].sum()) / 12,
    'Ingreso Mon. Neto Mensual S/.': ((x['INGMO2HD'] * x['FACTOR07']).sum() / x['FACTOR07'].sum()) / 12,
    'Tasa de Pobreza (%)': (x['ES_POBRE'] * x['FACTOR07']).sum() / x['FACTOR07'].sum() * 100,
    'Promedio Miembros Hogar': (x['MIEPERHO'] * x['FACTOR07']).sum() / x['FACTOR07'].sum()
})).reset_index()

# 3. Formateo de la tabla
pd.options.display.float_format = '{:,.2f}'.format
print("### RESUMEN ANUAL: MÓDULO SUMARIA (DATOS EXPANDIDOS) ###")
display(resumen_sumaria_anual)

### RESUMEN ANUAL: MÓDULO SUMARIA (DATOS EXPANDIDOS) ###


/tmp/ipykernel_5655/3329690101.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  resumen_sumaria_anual = enaho_sumaria_total.groupby('AÑO').apply(lambda x: pd.Series({


,AÑO,Hogares Totales (Est.),Ingreso Neto Anual S/. (Promedio),Ingreso Neto Mensual S/. (Promedio),Ingreso Mon. Neto Mensual S/.,Tasa de Pobreza (%),Promedio Miembros Hogar
0,2020,"9,394,191.85","29,806.31","2,483.86","1,826.70",23.27,3.51
1,2021,"9,903,824.11","33,690.54","2,807.54","2,115.32",20.02,3.36
2,2022,"9,999,295.18","37,230.28","3,102.52","2,363.50",21.90,3.38
3,2023,"10,196,775.39","38,862.96","3,238.58","2,477.07",23.15,3.34
4,2024,"10,360,811.18","40,315.63","3,359.64","2,576.59",21.85,3.33


## 3.5 Módulo 200 — Características del individuo

| Variable  | Descripción | Categorías / Missing |
|-----------|-------------|--------------------|
| `P207`    | Sexo | 1=Hombre, 2=Mujer |
| `P208A`   | Edad en años cumplidos | Numérica; **99=missing** |
| `ESTRATO` | Estrato geográfico | 1–6=Urbano, 7–8=Rural |
| `DOMINIO` | Dominio geográfico | 1–3=Costa, 4–6=Sierra, 7=Selva, 8=Lima |

> `ESTRATO` y `DOMINIO` están en el Módulo 200 — no se necesita Sumaria.

In [ ]:
# 1. Autenticación: Al ejecutar esto, aparecerá una ventana emergente.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive_api = GoogleDrive(gauth)

# 1. Diccionario con los IDs de Características de Miembros
archivos_dem = {
    "2024": "1ploeoUT3ebHmnmrpeC797uJqApUO5wNF",
    "2023": "1eOBT1tdbL9WgHTBXn6__oGcC3ySwZ_DQ",
    "2022": "1CzZTsbxUvyyxnm8ntVQyhyRdY3b2wRJL",
    "2021": "1lvo-3GcJlc1yixyhQ63QL7IFSMiEXBNP",
    "2020": "1syRqiCh9aEv10HopG3R7n_kJ4bEBEAAj"
}

lista_años_dem = []

for anio, file_id in archivos_dem.items():
    print(f"--- Procesando Miembros año {anio} ---")
    try:
        archivo_descargado = drive_api.CreateFile({'id': file_id})
        contenido = archivo_descargado.GetContentString(encoding='latin-1')
        df = pd.read_csv(io.StringIO(contenido), low_memory=False)

        # Normalizar a minúsculas para el filtro inicial
        df.columns = df.columns.str.lower()

        # Variables solicitadas + UBIGEO para el cálculo de departamento
        cols_dem = ['conglome', 'vivienda', 'hogar', 'codperso', 'ubigeo', 'p207', 'p208a', 'estrato', 'dominio']

        # Filtrar existentes y estandarizar a MAYÚSCULAS
        todas_las_cols = [c for c in cols_dem if c in df.columns]
        dem_sel = df[todas_las_cols].copy()
        dem_sel.columns = dem_sel.columns.str.upper()

        # Limpieza de llaves
        for col in ['CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO']:
            dem_sel[col] = dem_sel[col].astype(str).str.strip()

        dem_sel['AÑO'] = anio
        lista_años_dem.append(dem_sel)
        print(f"Éxito: {anio} procesado ({len(dem_sel)} filas).")

    except Exception as e:
        print(f"Error en {anio}: {e}")

if lista_años_dem:
    enaho_dem_total = pd.concat(lista_años_dem, ignore_index=True)
    print("\n¡CONSOLIDACIÓN COMPLETA!")

--- Procesando Miembros año 2024 ---
Éxito: 2024 procesado (117721 filas).
--- Procesando Miembros año 2023 ---
Éxito: 2023 procesado (119747 filas).
--- Procesando Miembros año 2022 ---
Éxito: 2022 procesado (121253 filas).
--- Procesando Miembros año 2021 ---
Éxito: 2021 procesado (121067 filas).
--- Procesando Miembros año 2020 ---
Éxito: 2020 procesado (126831 filas).

¡CONSOLIDACIÓN COMPLETA!


In [ ]:
# Copia de trabajo
dem_clean = enaho_dem_total.copy()

# --- PARTE A: Limpieza de variables demográficas ---
dem_clean['P207']    = pd.to_numeric(dem_clean['P207'],    errors='coerce')
dem_clean['P208A']   = pd.to_numeric(dem_clean['P208A'],   errors='coerce').replace(99, np.nan)
dem_clean['ESTRATO'] = pd.to_numeric(dem_clean['ESTRATO'], errors='coerce')
dem_clean['DOMINIO'] = pd.to_numeric(dem_clean['DOMINIO'], errors='coerce')

# --- PARTE B: Creación de la variable DEPARTAMENTO (Cálculo regional) ---
# En Python, el equivalente a tu código de Stata (ubigeo/10000)
# es tomar los dos primeros caracteres del string del UBIGEO.
dem_clean['region'] = dem_clean['UBIGEO'].astype(str).str.zfill(6).str[:2].astype(int)

# Diccionario de etiquetas (equivalente a label define)
labels_dep = {
    1: "Amazonas", 2: "Ancash", 3: "Apurimac", 4: "Arequipa", 5: "Ayacucho",
    6: "Cajamarca", 7: "Callao", 8: "Cusco", 9: "Huancavelica", 10: "Huanuco",
    11: "Ica", 12: "Junin", 13: "La Libertad", 14: "Lambayeque", 15: "Lima",
    16: "Loreto", 17: "Madre de Dios", 18: "Moquegua", 19: "Pasco", 20: "Piura",
    21: "Puno", 22: "San Martín", 23: "Tacna", 24: "Tumbes", 25: "Ucayali"
}

# Crear columna con nombres de departamento (equivalente a label values)
dem_clean['DEPARTAMENTO_NOM'] = dem_clean['region'].map(labels_dep)

# --- REPORTE FINAL ---
print(f"### RESUMEN DEMOGRÁFICO ###")
print(f"Edad rango: {dem_clean['P208A'].min():.0f} a {dem_clean['P208A'].max():.0f} años")
print(f"Departamentos procesados: {dem_clean['DEPARTAMENTO_NOM'].nunique()}")

# Mostrar distribución por año
tabla_anual_dem = dem_clean.groupby('AÑO').agg({
    'CODPERSO': 'count',
    'P208A': 'mean'
}).rename(columns={'CODPERSO': 'Nro Personas', 'P208A': 'Edad Promedio'})

display(tabla_anual_dem)
display(dem_clean[['AÑO', 'UBIGEO', 'region', 'DEPARTAMENTO_NOM']].head())

enaho_dem_total = dem_clean

### RESUMEN DEMOGRÁFICO ###
Edad rango: 0 a 98 años
Departamentos procesados: 25


,Nro Personas,Edad Promedio
AÑO,,
2020,126831,33.32
2021,121067,33.67
2022,121253,33.85
2023,119747,34.17
2024,117721,34.90


,AÑO,UBIGEO,region,DEPARTAMENTO_NOM
0,2024,150509,15,Lima
1,2024,150509,15,Lima
2,2024,150509,15,Lima
3,2024,150509,15,Lima
4,2024,150509,15,Lima


## 3.6 Merge y filtros de muestra

In [ ]:
# 1. Definición de las llaves de unión (incluyendo el año)
llaves_persona = ['CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO', 'AÑO']
llaves_hogar = ['CONGLOME', 'VIVIENDA', 'HOGAR', 'AÑO'] # Sumaria es a nivel hogar

print("Iniciando el proceso de Merge...")

# Asegurar que las columnas de enaho_salud_total estén en mayúsculas para el merge
enaho_salud_total.columns = enaho_salud_total.columns.str.upper()

# Convertir las columnas clave de enaho_salud_total a tipo string para asegurar la compatibilidad en el merge
for col in ['CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO']:
    if col in enaho_salud_total.columns:
        enaho_salud_total[col] = enaho_salud_total[col].astype(str).str.strip()

# 2. Merge 1: Miembros (Base) + Salud
# Usamos how='left' para no perder a las personas que no reportaron gastos de salud
enaho_final = pd.merge(enaho_dem_total, enaho_salud_total,
                       on=llaves_persona,
                       how='left',
                       suffixes=('', '_salud'))

print(f"Post Salud: {enaho_final.shape}")

# 3. Merge 2: Base + Educación
# Nota: Si en tu base de educación no tienes CODPERSO, avísame para ajustarlo
enaho_final = pd.merge(enaho_final, enaho_edu_total,
                       on=llaves_persona,
                       how='left',
                       suffixes=('', '_edu'))

print(f"Post Educación: {enaho_final.shape}")

# 4. Merge 3: Base + Sumaria
# OJO: Sumaria no tiene 'CODPERSO' porque los datos son por HOGAR.
# Al unir por llaves_hogar, los ingresos se repetirán para cada miembro del mismo hogar.
enaho_final = pd.merge(enaho_final, enaho_sumaria_total,
                       on=llaves_hogar,
                       how='left',
                       suffixes=('', '_sumaria'))

print(f"Post Sumaria: {enaho_final.shape}")

# 5. Limpieza Post-Merge: Eliminar columnas duplicadas si las hay
# A veces el UBIGEO o FACTOR07 vienen en todas las bases
columnas_duplicadas = [c for c in enaho_final.columns if '_salud' in c or '_edu' in c or '_sumaria' in c]
# Si quieres mantener una base limpia, podrías eliminarlas, pero por ahora las dejamos.

# --- CREACIÓN DE INDICADORES FINALES ---

# A. Gasto de bolsillo como % del ingreso total del hogar (Anualizado)
# Evitamos división por cero con np.where
enaho_final['pct_gasto_salud_hogar'] = np.where(enaho_final['INGHOG2D'] > 0,
                                               (enaho_final['GASTO_BOLSILLO'] / enaho_final['INGHOG2D']) * 100,
                                               0)

# B. Clasificación de Gasto Catastrófico (ejemplo: si gasta > 10% del ingreso anual)
enaho_final['gasto_catastrofico'] = (enaho_final['pct_gasto_salud_hogar'] > 10).astype(int)

print("\n" + "="*30)
print("¡MERGE FINALIZADO!")
print(f"Registros totales: {len(enaho_final)}")
display(enaho_final[['AÑO', 'DEPARTAMENTO_NOM', 'P208A', 'GASTO_BOLSILLO', 'INGHOG2D', 'pct_gasto_salud_hogar']].head())

Iniciando el proceso de Merge...
Post Salud: (606619, 121)
Post Educación: (606619, 128)
Post Sumaria: (606619, 140)

¡MERGE FINALIZADO!
Registros totales: 606619


,AÑO,DEPARTAMENTO_NOM,P208A,GASTO_BOLSILLO,INGHOG2D,pct_gasto_salud_hogar
0,2024,Lima,74.00,"1,867.00","45,736.13",4.08
1,2024,Lima,70.00,0.00,"45,736.13",0.00
2,2024,Lima,22.00,0.00,"45,736.13",0.00
3,2024,Lima,19.00,0.00,"45,736.13",0.00
4,2024,Lima,48.00,0.00,"40,958.33",0.00


In [ ]:
# 1. Asegurar estandarización a MAYÚSCULAS en la base integrada
enaho_final.columns = enaho_final.columns.str.upper()

# Ensure column names are unique by keeping the first occurrence if duplicates exist
# This is a defensive step against potential duplicate column names after merges and .str.upper()
enaho_final = enaho_final.loc[:, ~enaho_final.columns.duplicated()]

# 2. Variables críticas para el cálculo (incluyendo las nuevas de educación y sexo)
# Nota: 'EDU_SUPERIOR' viene de tu procesamiento previo en el módulo de educación
vars_calculo = [
    'GASTO_BOLSILLO', 'INGHOG2D', 'TIENE_SEGURO', 'ENFERMO_4SEM',
    'ES_POBRE', 'FACTOR07', 'P208A', 'P207', 'EDU_SUPERIOR'
]

for v in vars_calculo:
    if v in enaho_final.columns:
        enaho_final[v] = pd.to_numeric(enaho_final[v], errors='coerce').fillna(0)

# 3. Creación del Consolidado Final (Agrupado por AÑO y DEPARTAMENTO_NOM)
consolidado_final = enaho_final.groupby(['AÑO', 'DEPARTAMENTO_NOM']).apply(lambda x: pd.Series({
    # --- DEMOGRAFÍA ---
    'POBLACION_ESTIMADA': x['FACTOR07'].sum(),
    'EDAD_PROMEDIO': (x['P208A'] * x['FACTOR07']).sum() / x['FACTOR07'].sum(),

    # SEXO: P207 (1=Hombre, 2=Mujer). Calculamos % de Mujeres ponderado
    'PCT_MUJERES': (x[x['P207'] == 2]['FACTOR07'].sum() / x['FACTOR07'].sum()) * 100,

    # --- EDUCACIÓN (Usando tu variable edu_superior) ---
    'PCT_EDU_SUPERIOR': (x[x['EDU_SUPERIOR'] == 1]['FACTOR07'].sum() / x['FACTOR07'].sum()) * 100,

    # --- POBREZA ---
    'TASA_POBREZA_PCT': (x['ES_POBRE'] * x['FACTOR07']).sum() / x['FACTOR07'].sum() * 100,

    # --- SALUD ---
    'COBERTURA_SEGURO_PCT': (x['TIENE_SEGURO'] * x['FACTOR07']).sum() / x['FACTOR07'].sum() * 100,
    'MORBILIDAD_4SEM_PCT': (x['ENFERMO_4SEM'] * x['FACTOR07']).sum() / x['FACTOR07'].sum() * 100,
    'GASTO_BOLSILLO_PROM_S/': (x['GASTO_BOLSILLO'] * x['FACTOR07']).sum() / x['FACTOR07'].sum(),

    # --- ECONOMÍA ---
    'INGRESO_HOGAR_ANUAL_PROM': (x['INGHOG2D'] * x['FACTOR07']).sum() / x['FACTOR07'].sum(),
    'PESO_SALUD_EN_INGRESO_PCT': ((x['GASTO_BOLSILLO'] * x['FACTOR07']).sum() /
                                  (x['INGHOG2D'] * x['FACTOR07']).sum()) * 100
})).reset_index()

# 4. Ordenar y Formatear
consolidado_final = consolidado_final.sort_values(by=['AÑO', 'DEPARTAMENTO_NOM'])
pd.options.display.float_format = '{:,.2f}'.format

# 5. Mostrar Resultados
print("### CONSOLIDADO FINAL: DEMOGRAFÍA, EDUCACIÓN, SALUD Y POBREZA (PONDERADO) ###")
display(consolidado_final.head(10))

### CONSOLIDADO FINAL: DEMOGRAFÍA, EDUCACIÓN, SALUD Y POBREZA (PONDERADO) ###


/tmp/ipykernel_5655/3082358321.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  consolidado_final = enaho_final.groupby(['AÑO', 'DEPARTAMENTO_NOM']).apply(lambda x: pd.Series({


,AÑO,DEPARTAMENTO_NOM,POBLACION_ESTIMADA,EDAD_PROMEDIO,PCT_MUJERES,PCT_EDU_SUPERIOR,TASA_POBREZA_PCT,COBERTURA_SEGURO_PCT,MORBILIDAD_4SEM_PCT,GASTO_BOLSILLO_PROM_S/,INGRESO_HOGAR_ANUAL_PROM,PESO_SALUD_EN_INGRESO_PCT
0,2020,Amazonas,"439,922.51",31.73,50.14,6.73,35.73,88.05,48.56,116.75,"24,658.00",0.47
1,2020,Ancash,"1,198,778.40",33.09,51.48,9.86,29.63,82.61,73.27,127.83,"28,040.04",0.46
2,2020,Apurimac,"487,148.18",33.88,50.61,9.62,35.35,90.94,47.29,68.35,"23,683.50",0.29
3,2020,Arequipa,"1,373,797.58",34.09,50.69,16.77,18.51,63.69,45.94,163.97,"37,265.61",0.44
4,2020,Ayacucho,"732,932.00",32.50,52.16,7.59,46.19,88.36,58.78,71.60,"24,250.94",0.30
5,2020,Cajamarca,"1,549,996.30",34.39,51.88,6.24,42.54,86.49,56.24,121.68,"21,265.89",0.57
6,2020,Callao,"1,092,257.31",34.35,52.60,11.86,35.04,76.57,36.78,152.07,"37,048.14",0.41
7,2020,Cusco,"1,371,028.01",33.61,51.78,8.47,32.22,82.29,41.80,118.02,"22,544.83",0.52
8,2020,Huancavelica,"517,355.78",32.96,52.47,5.53,47.15,93.92,61.37,43.78,"17,235.03",0.25
9,2020,Huanuco,"905,064.20",32.19,50.47,8.89,42.64,86.90,60.00,90.29,"22,080.81",0.41


## 3.7 Índice de Desarrollo Humano (2020-2024)

**Índice de Desarrollo Humano (IDH)**

Este índice está compuesto por factores sanitarios, educativos y económicos para medir el desarrollo humano que se tiene en un país, a diferencia de las miradas tradicionales que
medían el desarrollo en términos meramente económicos (PNUD).

| Dimensión | Indicadores | Índice |
|-----------|-------------|--------|
| **Salud** | Esperanza de vida al nacer | Índice de esperanza de vida |
| **Educación** | Años esperados de escolaridad · Años promedio de escolaridad | Índice de educación |
| **Nivel de vida** | Ingreso real per cápita del hogar (S/) | Índice de ingresos |

In [ ]:
# Vinculamos con ID único del archivo en Google Drive
FILE_ID_IDH = "1GnJZntyyqvMD9ghRu0YLh48zsEDzMJsZ"

archivo_idh = drive_api.CreateFile({"id": FILE_ID_IDH})
archivo_idh.GetContentFile("idh_2017_2024.xlsx")
print("Archivo IDH descargado correctamente")

# Leemos el excel con pandas
df_idh_raw = pd.read_excel(
    "idh_2017_2024.xlsx",
    sheet_name="Anexo I -IDH",
    header=None  # Aquí leemos sin encabezado para manejarlo manualmente
)

print("Dimensiones del archivo raw:", df_idh_raw.shape)
df_idh_raw.head(6)

Archivo IDH descargado correctamente
Dimensiones del archivo raw: (2101, 58)


,0,1,2,3,4,5,6,7,8,9,...,48,49,50,51,52,53,54,55,56,57
0,NaN,NaN,2017,NaN,NaN,NaN,NaN,NaN,NaN,2018,...,NaN,NaN,NaN,2024,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Población,Esperanza de Vida al Nacer,Valor esperado de años de educación en el hogar,Años de educación,Ingreso real per cápita del hogar,IDH,Ranking,Población,...,Ingreso real per cápita del hogar,IDH,Ranking,Población,Esperanza de Vida al Nacer,Valor esperado de años de educación en el hogar,Años de educación,Ingreso real per cápita del hogar,IDH,Ranking
2,Ubigeo,DEPARTAMENTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,PROVINCIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,DISTRITO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,000000,PERÚ,31237384,76.62,0.65,8.99,952.04,0.65,NaN,31562130,...,892.10,0.66,NaN,34038456,78.33,0.67,9.45,896.54,0.66,NaN


In [ ]:
# Los años están en la fila 0, las variables en la fila 1
# Las columnas 0 y 1 son Ubigeo y Departamento/Provincia/Distrito

años_row   = df_idh_raw.iloc[0]   # fila con los años
vars_row   = df_idh_raw.iloc[1]   # fila con nombres de variables

# Construimos los nombres de columna combinando año + variable
columnas = []
año_actual = None
for i, (anio, var) in enumerate(zip(años_row, vars_row)):
    if pd.notna(anio) and str(anio).strip() not in ['None', '']:
        año_actual = int(float(str(anio)))
    if i == 0:
        columnas.append("ubigeo")
    elif i == 1:
        columnas.append("nombre")
    else:
        if pd.notna(var) and año_actual is not None:
            columnas.append(f"{var.strip()}_{año_actual}")
        else:
            columnas.append(f"col_{i}")

df_idh_raw.columns = columnas

# Nos quedamos solo con datos (desde fila 5 en adelante)
df_idh = df_idh_raw.iloc[5:].reset_index(drop=True)

print("Shape después de limpiar encabezados:", df_idh.shape)
df_idh.head()

Shape después de limpiar encabezados: (2096, 58)


,ubigeo,nombre,Población_2017,Esperanza de Vida al Nacer_2017,Valor esperado de años de educación en el hogar_2017,Años de educación_2017,Ingreso real per cápita del hogar_2017,IDH_2017,Ranking_2017,Población_2018,...,Ingreso real per cápita del hogar_2023,IDH_2023,Ranking_2023,Población_2024,Esperanza de Vida al Nacer_2024,Valor esperado de años de educación en el hogar_2024,Años de educación_2024,Ingreso real per cápita del hogar_2024,IDH_2024,Ranking_2024
0,000000,PERÚ,31237384,76.62,0.65,8.99,952.04,0.65,NaN,31562130,...,892.10,0.66,NaN,34038456,78.33,0.67,9.45,896.54,0.66,NaN
1,010000,AMAZONAS,417365,76.06,0.56,6.86,578.20,0.53,NaN,419833,...,635.89,0.56,NaN,430123,75.32,0.59,7.55,632.74,0.55,NaN
2,010100,CHACHAPOYAS,"60,419.00",76.44,0.59,7.99,754.18,0.59,NaN,61078,...,839.41,0.61,NaN,64718,76.72,0.62,8.35,859.22,0.62,NaN
3,010101,CHACHAPOYAS,35868,76.41,0.66,9.23,944.37,0.65,191,36602,...,995.00,0.66,182,41335,76.69,0.70,9.06,"1,009.36",0.67,184
4,010102,ASUNCION,283.00,76.41,0.47,5.57,433.67,0.46,1127,283,...,535.53,0.51,1004,271,76.66,0.37,6.66,476.72,0.47,1351


In [ ]:
# Filtramos la información solo a nivel departamental, retiramos a PERÚ con código 000000

df_dpto = df_idh[
    df_idh["ubigeo"].astype(str).str.match(r'^\d{2}0000$')
].copy()

df_dpto = df_dpto[df_dpto["ubigeo"] != "000000"].reset_index(drop=True)

print(f"Departamentos encontrados: {len(df_dpto)}")
print(df_dpto[["ubigeo", "nombre"]].head(26))

Departamentos encontrados: 25
    ubigeo         nombre
0   010000       AMAZONAS
1   020000         ANCASH
2   030000       APURIMAC
3   040000       AREQUIPA
4   050000       AYACUCHO
5   060000      CAJAMARCA
6   070000         CALLAO
7   080000          CUSCO
8   090000   HUANCAVELICA
9   100000        HUANUCO
10  110000            ICA
11  120000          JUNIN
12  130000    LA LIBERTAD
13  140000     LAMBAYEQUE
14  150000           LIMA
15  160000         LORETO
16  170000  MADRE DE DIOS
17  180000       MOQUEGUA
18  190000          PASCO
19  200000          PIURA
20  210000           PUNO
21  220000     SAN MARTIN
22  230000          TACNA
23  240000         TUMBES
24  250000        UCAYALI


In [ ]:
# Filtramos solo columnas de IDH por año
# De cada año nos quedamos con: ubigeo, nombre e IDH

cols_idh = ["ubigeo", "nombre"] + [c for c in df_dpto.columns if c.startswith("IDH_")]

df_idh_dpto = df_dpto[cols_idh].copy()

# Renombramos las columnas IDH para mayor claridad
df_idh_dpto.columns = (
    ["ubigeo", "departamento"] +
    [f"idh_{c.split('_')[1]}" for c in cols_idh[2:]]
)

# Convertimos las columnas IDH a numeric
for col in df_idh_dpto.columns:
    if col.startswith('idh_'):
        df_idh_dpto[col] = pd.to_numeric(df_idh_dpto[col], errors='coerce') # Use coerce to handle any non-numeric values

print(df_idh_dpto.head())
print("\nTipos de datos:")
print(df_idh_dpto.dtypes)

   ubigeo departamento  idh_2017  idh_2018  idh_2019  idh_2020  idh_2021  \
0  010000     AMAZONAS      0.53      0.54      0.55      0.53      0.53   
1  020000       ANCASH      0.59      0.59      0.61      0.55      0.56   
2  030000     APURIMAC      0.50      0.51      0.55      0.51      0.52   
3  040000     AREQUIPA      0.71      0.72      0.71      0.70      0.71   
4  050000     AYACUCHO      0.51      0.53      0.53      0.52      0.52   

   idh_2022  idh_2023  idh_2024  
0      0.55      0.56      0.55  
1      0.61      0.61      0.62  
2      0.56      0.58      0.58  
3      0.73      0.72      0.74  
4      0.56      0.57      0.57  

Tipos de datos:
ubigeo           object
departamento     object
idh_2017        float64
idh_2018        float64
idh_2019        float64
idh_2020        float64
idh_2021        float64
idh_2022        float64
idh_2023        float64
idh_2024        float64
dtype: object


In [ ]:
# Convertimos a formato largo para mayor practicidad cuando se haga el merge con la bbdd ENAHO

df_idh_long = df_idh_dpto.melt(
    id_vars=["ubigeo", "departamento"],
    var_name="año",
    value_name="idh"
)

df_idh_long["año"] = df_idh_long["año"].str.replace("idh_", "").astype(int)
df_idh_long["ubigeo_dpto"] = df_idh_long["ubigeo"].astype(str).str[:2]

print("Shape formato largo:", df_idh_long.shape)
print(df_idh_long.head(10))

Shape formato largo: (200, 5)
   ubigeo  departamento   año  idh ubigeo_dpto
0  010000      AMAZONAS  2017 0.53          01
1  020000        ANCASH  2017 0.59          02
2  030000      APURIMAC  2017 0.50          03
3  040000      AREQUIPA  2017 0.71          04
4  050000      AYACUCHO  2017 0.51          05
5  060000     CAJAMARCA  2017 0.50          06
6  070000        CALLAO  2017 0.73          07
7  080000         CUSCO  2017 0.56          08
8  090000  HUANCAVELICA  2017 0.44          09
9  100000       HUANUCO  2017 0.53          10


## 3.8 Exportación de bases limpias

In [ ]:
# 1. Preparación del DataFrame IDH
# Aseguramos que los nombres estén en MAYÚSCULAS para consistencia
df_idh_long.columns = df_idh_long.columns.str.upper()

# Convertimos UBIGEO_DPTO a entero para que coincida con la variable 'REGION' o el código extraído
df_idh_long['UBIGEO_DPTO'] = df_idh_long['UBIGEO_DPTO'].astype(int)

# 2. Preparación del Consolidado Final para el Merge

# --- Diagnóstico y Conversión de tipos para la columna 'AÑO' ---
print(f"Tipo de consolidado_final['AÑO'] antes de la conversión: {consolidado_final['AÑO'].dtype}")
print(f"Tipo de df_idh_long['AÑO'] antes del merge: {df_idh_long['AÑO'].dtype}")

# Convertir la columna 'AÑO' en consolidado_final a int para que coincida con df_idh_long
consolidado_final['AÑO'] = consolidado_final['AÑO'].astype(int)

# Estandarizar el casing de los nombres de departamento en consolidado_final para el merge
consolidado_final['DEPARTAMENTO_NOM'] = consolidado_final['DEPARTAMENTO_NOM'].str.upper()

print(f"Tipo de consolidado_final['AÑO'] después de la conversión: {consolidado_final['AÑO'].dtype}")

# 3. Ejecución del MERGE
# Unimos el consolidado con el IDH
consolidado_completo_idh = pd.merge(
    consolidado_final,
    df_idh_long[['AÑO', 'DEPARTAMENTO', 'IDH']],
    left_on=['AÑO', 'DEPARTAMENTO_NOM'],
    right_on=['AÑO', 'DEPARTAMENTO'],
    how='left'
)

# 4. Limpieza post-merge
# Eliminamos la columna repetida 'DEPARTAMENTO'
if 'DEPARTAMENTO' in consolidado_completo_idh.columns:
    consolidado_completo_idh = consolidado_completo_idh.drop(columns=['DEPARTAMENTO'])

# 5. Formateo y Visualización Final
pd.options.display.float_format = '{:,.3f}'.format # 3 decimales para el IDH
print("### CONSOLIDADO ENAHO + IDH (2020-2024) ###")
display(consolidado_completo_idh.head(10))

# --- ANÁLISIS RÁPIDO ---
# ¿Existe correlación entre el IDH y el Gasto de Bolsillo?
correlacion = consolidado_completo_idh[['IDH', 'GASTO_BOLSILLO_PROM_S/', 'TASA_POBREZA_PCT']].corr()
print("\nMatriz de Correlación:")
display(correlacion)

Tipo de consolidado_final['AÑO'] antes de la conversión: object
Tipo de df_idh_long['AÑO'] antes del merge: int64
Tipo de consolidado_final['AÑO'] después de la conversión: int64
### CONSOLIDADO ENAHO + IDH (2020-2024) ###


,AÑO,DEPARTAMENTO_NOM,POBLACION_ESTIMADA,EDAD_PROMEDIO,PCT_MUJERES,PCT_EDU_SUPERIOR,TASA_POBREZA_PCT,COBERTURA_SEGURO_PCT,MORBILIDAD_4SEM_PCT,GASTO_BOLSILLO_PROM_S/,INGRESO_HOGAR_ANUAL_PROM,PESO_SALUD_EN_INGRESO_PCT,IDH
0,2020,AMAZONAS,"439,922.511",31.725,50.140,6.726,35.731,88.054,48.564,116.754,"24,658.003",0.473,0.529
1,2020,ANCASH,"1,198,778.404",33.087,51.481,9.862,29.630,82.613,73.270,127.826,"28,040.038",0.456,0.550
2,2020,APURIMAC,"487,148.176",33.885,50.614,9.622,35.351,90.945,47.288,68.346,"23,683.496",0.289,0.507
3,2020,AREQUIPA,"1,373,797.581",34.090,50.690,16.772,18.507,63.688,45.938,163.973,"37,265.608",0.440,0.697
4,2020,AYACUCHO,"732,932.001",32.497,52.157,7.593,46.186,88.359,58.776,71.604,"24,250.937",0.295,0.523
5,2020,CAJAMARCA,"1,549,996.303",34.391,51.878,6.242,42.536,86.492,56.240,121.681,"21,265.890",0.572,0.498
6,2020,CALLAO,"1,092,257.311",34.352,52.603,11.858,35.041,76.575,36.784,152.070,"37,048.141",0.410,0.688
7,2020,CUSCO,"1,371,028.006",33.614,51.783,8.472,32.224,82.285,41.801,118.019,"22,544.828",0.523,0.499
8,2020,HUANCAVELICA,"517,355.780",32.961,52.468,5.526,47.151,93.916,61.367,43.784,"17,235.029",0.254,0.477
9,2020,HUANUCO,"905,064.200",32.189,50.473,8.885,42.644,86.898,59.995,90.292,"22,080.805",0.409,0.523



Matriz de Correlación:


,IDH,GASTO_BOLSILLO_PROM_S/,TASA_POBREZA_PCT
IDH,1.000,0.623,-0.685
GASTO_BOLSILLO_PROM_S/,0.623,1.000,-0.527
TASA_POBREZA_PCT,-0.685,-0.527,1.000


In [ ]:
enaho_final['COD_DPTO'] = enaho_final['COD_DPTO'].astype(str).str.zfill(2)
df_idh_long['UBIGEO_DPTO'] = df_idh_long['UBIGEO_DPTO'].astype(str).str.zfill(2)
enaho_final['AÑO'] = enaho_final['AÑO'].astype(int)
df_idh_long['AÑO'] = df_idh_long['AÑO'].astype(int)

base_final = enaho_final.merge(
    df_idh_long[['UBIGEO_DPTO', 'AÑO', 'IDH']],
    left_on=['COD_DPTO', 'AÑO'],
    right_on=['UBIGEO_DPTO', 'AÑO'],
    how='left'
)

In [ ]:
# Se monta Google Drive en Colab para poder acceder a archivos y carpetas
# almacenados en la cuenta personal de Drive. Esto permitirá guardar la base
# de datos final limpia y reutilizarla en otros notebooks del trabajo.

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Se crea una carpeta de trabajo en Google Drive, para
# almacenar la base final ya depurada. Luego, la base llamada `base_final`
# se guarda en formato .pkl, lo que permite conservar el dataframe y cargarlo
# fácilmente en otros notebooks, como el de estadística descriptiva.

import os

carpeta = '/content/drive/MyDrive/TC1'
os.makedirs(carpeta, exist_ok=True)

ruta_base = f'{carpeta}/base_final_limpia.pkl'
base_final.to_pickle(ruta_base)

print("Base guardada en:", ruta_base)
print("Dimensiones:", base_final.shape)

Base guardada en: /content/drive/MyDrive/TC1/base_final_limpia.pkl
Dimensiones: (606619, 144)
